# Otimização de Hiperparâmetros com AG para o pipeline de câncer

Este notebook adapta a lógica de `ga_hyperparam_optimization.ipynb` para o pipeline real de `analise_cancer.ipynb`. O objetivo é executar otimização de hiperparâmetros via Algoritmo Genético para modelos de diagnóstico, com foco em indicadores de saúde da mulher: sensibilidade (recall), especificidade, F1-score e equidade entre grupos demográficos.

O notebook inclui:
- carregamento e preparação do dataset do `analise_cancer.ipynb`
- baseline com todos os modelos originais (RandomForest, LogisticRegression, KNeighborsClassifier e ExtraTreesClassifier)
- codificação de genes para hiperparâmetros
- operadores AG (seleção, cruzamento, mutação)
- função de fitness ponderada
- 3 experimentos com parâmetros de AG
- comparação dos modelos otimizados contra baseline.

## Dependências

Instale as dependências necessárias caso ainda não estejam disponíveis no ambiente:
```bash
pip install -U scikit-learn pandas numpy matplotlib seaborn joblib
```
Este notebook utiliza um AG em Python puro, sem dependências extras para o mecanismo genético.

In [7]:
# Importar bibliotecas
import warnings
warnings.filterwarnings('ignore')

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

sns.set(style='whitegrid')
print('Bibliotecas importadas')

Bibliotecas importadas


## Carregar e preparar o dataset

Aqui usamos o mesmo dataset de câncer de pulmão do `analise_cancer.ipynb`. Se o arquivo não estiver no caminho esperado, o notebook gera um fallback sintético para demonstração.

In [8]:
dataset_path = 'datasets/dataset.csv'
if os.path.exists(dataset_path):
    df = pd.read_csv(dataset_path)
    print(f'Dataset carregado: {dataset_path} -> {df.shape[0]} linhas')
else:
    print(f'Dataset não encontrado em {dataset_path}. Gerando dataset sintético para demonstração.')
    rng = np.random.RandomState(42)
    n = 500
    df = pd.DataFrame({
        'GENDER': rng.choice(['M', 'F'], size=n, p=[0.5, 0.5]),
        'AGE': rng.randint(18, 85, size=n),
        'SMOKING': rng.choice([1,2], size=n, p=[0.7,0.3]),
        'YELLOW FINGERS': rng.choice([1,2], size=n, p=[0.8,0.2]),
        'CHRONIC DISEASE': rng.choice([1,2], size=n, p=[0.85,0.15]),
        'ALCOHOL CONSUMING': rng.choice([1,2], size=n, p=[0.9,0.1]),
        'COUGHING': rng.choice([1,2], size=n, p=[0.8,0.2]),
    })
    logits = (df['SMOKING'] == 2).astype(int) * 0.6 + (df['AGE'] > 60).astype(int) * 0.3 + rng.normal(0, 0.2, size=n)
    df['LUNG_CANCER'] = np.where(logits > 0.5, 'YES', 'NO')

target_column = 'LUNG_CANCER'
demographic_column = 'GENDER'
df.head()

Dataset carregado: datasets/dataset.csv -> 3000 linhas


,GENDER,AGE,SMOKING,YELLOW_FINGERS,ANXIETY,PEER_PRESSURE,CHRONIC_DISEASE,FATIGUE,ALLERGY,WHEEZING,ALCOHOL_CONSUMING,COUGHING,SHORTNESS_OF_BREATH,SWALLOWING_DIFFICULTY,CHEST_PAIN,LUNG_CANCER
0,M,65,1,1,1,2,2,1,2,2,2,2,2,2,1,NO
1,F,55,1,2,2,1,1,2,2,2,1,1,1,2,2,NO
2,F,78,2,2,1,1,1,2,1,2,1,1,2,1,1,YES
3,M,60,2,1,1,1,2,1,2,1,1,2,1,2,2,YES
4,F,80,1,1,2,1,1,2,1,2,1,1,1,1,2,NO


## Pré-processamento e split

Montamos o pipeline de pré-processamento igual ao do `analise_cancer.ipynb` e separamos treino, validação e teste com estratificação.

In [9]:
X = df.drop(columns=[target_column])
y = df[target_column]
if y.dtype == object or y.dtype == "string" or y.dtype.name == "category":
    y = y.astype(str).str.strip().str.upper().map({"NO": 0, "YES": 1})
    if y.isna().any():
        y = df[target_column].astype("category").cat.codes



## Métricas de saúde e equidade

Definimos funções para sensibilidade, especificidade, F1-score e uma medida simples de equidade baseada na diferença de recall entre grupos.

In [10]:
def sensitivity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tp / (tp + fn) if (tp + fn) > 0 else 0

def specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else 0

def f1_score_custom(y_true, y_pred):
    return f1_score(y_true, y_pred, zero_division=0)

def group_fairness_score(y_true, y_pred, groups):
    dfm = pd.DataFrame({'y_true': y_true, 'y_pred': y_pred, 'group': groups})
    recalls = dfm.groupby('group').apply(lambda g: recall_score(g['y_true'], g['y_pred'], zero_division=0))
    return float(recalls.max() - recalls.min())

print('Métricas definidas')

Métricas definidas


## Baseline com modelos originais

Treinamos cada modelo original sem otimização para medir a melhoria obtida pelo AG.

In [11]:
colunas_categoricas = X.select_dtypes(include=['object', 'string']).columns.tolist()
colunas_numericas = [col for col in X.columns if col not in colunas_categoricas]
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), colunas_categoricas),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), colunas_numericas)
    ]
)
# Train / validation / test split
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)
print('Shapes:', X_train.shape, X_val.shape, X_test.shape)


Shapes: (1800, 15) (600, 15) (600, 15)


In [12]:
original_models = {
    'RandomForestClassifier': RandomForestClassifier(random_state=42, class_weight='balanced'),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'KNeighborsClassifier': KNeighborsClassifier(),
    'ExtraTreesClassifier': ExtraTreesClassifier(random_state=42, class_weight='balanced')
}

baseline_results = {}
for model_name, clf in original_models.items():
    pipeline = Pipeline([('pre', preprocessor), ('clf', clf)])
    pipeline.fit(X_train, y_train)
    y_val_pred = pipeline.predict(X_val)
    y_test_pred = pipeline.predict(X_test)

    baseline_results[model_name] = {
        'Accuracy_val': accuracy_score(y_val, y_val_pred),
        'Recall_val': sensitivity(y_val, y_val_pred),
        'Specificity_val': specificity(y_val, y_val_pred),
        'F1_val': f1_score_custom(y_val, y_val_pred),
        'Accuracy_test': accuracy_score(y_test, y_test_pred),
        'Recall_test': sensitivity(y_test, y_test_pred),
        'Specificity_test': specificity(y_test, y_test_pred),
        'F1_test': f1_score_custom(y_test, y_test_pred)
    }

baseline_metrics = baseline_results['RandomForestClassifier']
baseline_results

{'RandomForestClassifier': {'Accuracy_val': 0.5366666666666666,
  'Recall_val': np.float64(0.5511551155115512),
  'Specificity_val': np.float64(0.5218855218855218),
  'F1_val': 0.545751633986928,
  'Accuracy_test': 0.5366666666666666,
  'Recall_test': np.float64(0.5493421052631579),
  'Specificity_test': np.float64(0.5236486486486487),
  'F1_test': 0.545751633986928},
 'LogisticRegression': {'Accuracy_val': 0.515,
  'Recall_val': np.float64(0.5445544554455446),
  'Specificity_val': np.float64(0.48484848484848486),
  'F1_val': 0.5314009661835749,
  'Accuracy_test': 0.52,
  'Recall_test': np.float64(0.5822368421052632),
  'Specificity_test': np.float64(0.4560810810810811),
  'F1_test': 0.5514018691588785},
 'KNeighborsClassifier': {'Accuracy_val': 0.54,
  'Recall_val': np.float64(0.5808580858085809),
  'Specificity_val': np.float64(0.4983164983164983),
  'F1_val': 0.5605095541401274,
  'Accuracy_test': 0.505,
  'Recall_test': np.float64(0.5164473684210527),
  'Specificity_test': np.float

## Operadores do Algoritmo Genético

Implementamos seleção por torneio, cruzamento uniforme e mutação por reset aleatório.

In [13]:
def sample_param(param_spec, rng):
    ptype = param_spec['type']
    if ptype == 'int':
        return rng.randint(param_spec['low'], param_spec['high'] + 1)
    elif ptype == 'float':
        return rng.uniform(param_spec['low'], param_spec['high'])
    elif ptype == 'cat':
        return rng.choice(param_spec['choices'])
    else:
        raise ValueError('Unknown param type')

def random_individual(param_space, rng):
    return {k: sample_param(spec, rng) for k, spec in param_space.items()}

def crossover(parent1, parent2, rng, cx_prob=0.8):
    child = {}
    for k in parent1:
        child[k] = parent1[k] if rng.random() < cx_prob else parent2[k]
    return child

def mutate(indiv, param_space, rng, mut_prob=0.1):
    for k, spec in param_space.items():
        if rng.random() < mut_prob:
            indiv[k] = sample_param(spec, rng)
    return indiv

def tournament_selection(pop, fitnesses, k=3, rng=None):
    rng = rng or random
    best_idx = rng.randint(0, len(pop)-1)
    for _ in range(k-1):
        idx = rng.randint(0, len(pop)-1)
        if fitnesses[idx] > fitnesses[best_idx]:
            best_idx = idx
    return pop[best_idx]

print('Operadores AG definidos')

Operadores AG definidos


## Função de fitness com prioridades clínicas

A função de fitness maximiza recall, especificidade e F1, e penaliza distância de recall entre grupos demográficos.

In [14]:
def evaluate_individual(indiv, estimator_cls, X_tr, y_tr, X_va, y_va, groups_va, weights, rng):
    params = {}
    for k, v in indiv.items():
        params[k] = v

    valid_keys = [name for name in estimator_cls.__init__.__code__.co_varnames if name != 'self']
    clf_params = {k: v for k, v in params.items() if k in valid_keys}
    clf = estimator_cls(**clf_params)
    pipeline = Pipeline([('pre', preprocessor), ('clf', clf)])
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_va)

    rec = sensitivity(y_va, y_pred)
    spec = specificity(y_va, y_pred)
    f1v = f1_score_custom(y_va, y_pred)
    fairness = group_fairness_score(y_va, y_pred, groups_va)

    w_rec, w_spec, w_f1, w_fair = weights
    fitness = (w_rec * rec) + (w_spec * spec) + (w_f1 * f1v) - (w_fair * fairness)
    return float(fitness), {'recall': rec, 'specificity': spec, 'f1': f1v, 'fairness': fairness}

print('Função de fitness definida')

Função de fitness definida


## Executor do AG

Função que executa o loop de geração, mantendo o melhor indivíduo e atualizando a população.

In [15]:
def run_ga_experiment(param_space, estimator_cls, X_tr, y_tr, X_va, y_va, groups_va, pop_size=20, generations=10, cx_prob=0.8, mut_prob=0.1, weights=(0.6,0.2,0.2,0.5), seed=42):
    rng = random.Random(seed)
    population = [random_individual(param_space, rng) for _ in range(pop_size)]
    fitnesses = [evaluate_individual(ind, estimator_cls, X_tr, y_tr, X_va, y_va, groups_va, weights, rng)[0] for ind in population]
    best = None
    best_score = -1e9
    history = []

    for gen in range(generations):
        if max(fitnesses) > best_score:
            best_score = max(fitnesses)
            best = population[int(np.argmax(fitnesses))]
        history.append(best_score)
        new_pop = [best]
        while len(new_pop) < pop_size:
            p1 = tournament_selection(population, fitnesses, k=3, rng=rng)
            p2 = tournament_selection(population, fitnesses, k=3, rng=rng)
            child = crossover(p1, p2, rng, cx_prob=cx_prob)
            child = mutate(child, param_space, rng, mut_prob=mut_prob)
            new_pop.append(child)
        population = new_pop
        fitnesses = [evaluate_individual(ind, estimator_cls, X_tr, y_tr, X_va, y_va, groups_va, weights, rng)[0] for ind in population]
        print(f'Gen {gen+1}/{generations} - melhor fitness: {best_score:.4f}')

    best_metrics = evaluate_individual(best, estimator_cls, X_tr, y_tr, X_va, y_va, groups_va, weights, rng)[1]
    return best, best_score, best_metrics, history

Definimos quatro modelos a serem otimizados: RandomForestClassifier, LogisticRegression, KNeighborsClassifier e ExtraTreesClassifier.

In [16]:
param_space_rf = {
    'n_estimators': {'type': 'int', 'low': 50, 'high': 250},
    'max_depth': {'type': 'int', 'low': 3, 'high': 30},
    'min_samples_split': {'type': 'int', 'low': 2, 'high': 10},
    'min_samples_leaf': {'type': 'int', 'low': 1, 'high': 4},
    'max_features': {'type': 'float', 'low': 0.3, 'high': 1.0}
}

param_space_lr = {
    'C': {'type': 'float', 'low': 0.01, 'high': 10.0},
    'penalty': {'type': 'cat', 'choices': ['l1', 'l2']},
    'solver': {'type': 'cat', 'choices': ['liblinear', 'saga']},
    'class_weight': {'type': 'cat', 'choices': [None, 'balanced']},
    'max_iter': {'type': 'int', 'low': 200, 'high': 1000}
}

param_space_knn = {
    'n_neighbors': {'type': 'int', 'low': 3, 'high': 15},
    'weights': {'type': 'cat', 'choices': ['uniform', 'distance']},
    'p': {'type': 'int', 'low': 1, 'high': 2}
}

param_space_et = {
    'n_estimators': {'type': 'int', 'low': 50, 'high': 250},
    'max_depth': {'type': 'int', 'low': 3, 'high': 30},
    'min_samples_split': {'type': 'int', 'low': 2, 'high': 10},
    'min_samples_leaf': {'type': 'int', 'low': 1, 'high': 4},
    'max_features': {'type': 'float', 'low': 0.3, 'high': 1.0}
}

model_spaces = {
    'RandomForestClassifier': (RandomForestClassifier, param_space_rf),
    'LogisticRegression': (LogisticRegression, param_space_lr),
    'KNeighborsClassifier': (KNeighborsClassifier, param_space_knn),
    'ExtraTreesClassifier': (ExtraTreesClassifier, param_space_et)
}

groups_val = X_val[demographic_column].values if demographic_column in X_val.columns else np.array(['all'] * len(X_val))

results = {}
for model_name, (estimator_cls, param_space) in model_spaces.items():
    print(f'### Executando experimentos para {model_name}')
    exp1 = run_ga_experiment(param_space, estimator_cls, X_train, y_train, X_val, y_val, groups_val,
                             pop_size=8, generations=3, cx_prob=0.8, mut_prob=0.1,
                             weights=(0.6,0.2,0.2,0.5), seed=1)

    exp2 = run_ga_experiment(param_space, estimator_cls, X_train, y_train, X_val, y_val, groups_val,
                             pop_size=8, generations=3, cx_prob=0.8, mut_prob=0.3,
                             weights=(0.6,0.2,0.2,0.5), seed=2)

    exp3 = run_ga_experiment(param_space, estimator_cls, X_train, y_train, X_val, y_val, groups_val,
                             pop_size=12, generations=3, cx_prob=0.8, mut_prob=0.1,
                             weights=(0.6,0.2,0.2,0.5), seed=3)

    results[model_name] = {'exp1': exp1, 'exp2': exp2, 'exp3': exp3}
    print(f'Fim de {model_name}: exp1={exp1[1]:.4f}, exp2={exp2[1]:.4f}, exp3={exp3[1]:.4f}')


### Executando experimentos para RandomForestClassifier
Gen 1/3 - melhor fitness: 0.5334
Gen 2/3 - melhor fitness: 0.5334
Gen 3/3 - melhor fitness: 0.5334
Gen 1/3 - melhor fitness: 0.5291
Gen 2/3 - melhor fitness: 0.5291
Gen 3/3 - melhor fitness: 0.5291
Gen 1/3 - melhor fitness: 0.5228
Gen 2/3 - melhor fitness: 0.5369
Gen 3/3 - melhor fitness: 0.5369
Fim de RandomForestClassifier: exp1=0.5334, exp2=0.5291, exp3=0.5369
### Executando experimentos para LogisticRegression
Gen 1/3 - melhor fitness: 0.5160
Gen 2/3 - melhor fitness: 0.5218
Gen 3/3 - melhor fitness: 0.5218
Gen 1/3 - melhor fitness: 0.5111
Gen 2/3 - melhor fitness: 0.5111
Gen 3/3 - melhor fitness: 0.5297
Gen 1/3 - melhor fitness: 0.5257
Gen 2/3 - melhor fitness: 0.5257
Gen 3/3 - melhor fitness: 0.5257
Fim de LogisticRegression: exp1=0.5218, exp2=0.5297, exp3=0.5257
### Executando experimentos para KNeighborsClassifier
Gen 1/3 - melhor fitness: 0.5440
Gen 2/3 - melhor fitness: 0.5440
Gen 3/3 - melhor fitness: 0.5440
Gen 1/3 - m

## Comparação entre modelos otimizados e modelos originais

Comparamos o desempenho de cada modelo otimizado com o seu modelo original treinado sem otimização no conjunto de teste.

In [17]:
def subgroup_metrics(pipe, X_te, y_te, group_col):
    df_te = X_te.reset_index(drop=True).copy()
    df_te['group'] = df_te[group_col]
    preds = pipe.predict(X_te)
    rows = []
    for g, subset in df_te.groupby('group'):
        idx = subset.index
        rows.append({
            'group': g,
            'recall': sensitivity(y_te.iloc[idx], preds[idx]),
            'specificity': specificity(y_te.iloc[idx], preds[idx]),
            'f1': f1_score_custom(y_te.iloc[idx], preds[idx])
        })
    return pd.DataFrame(rows)


def build_pipeline(estimator_cls, params):
    valid_keys = [name for name in estimator_cls.__init__.__code__.co_varnames if name != 'self']
    clf_params = {k: v for k, v in params.items() if k in valid_keys}
    clf = estimator_cls(**clf_params)
    return Pipeline([('pre', preprocessor), ('clf', clf)])


def evaluate_test_pipeline(pipe, X_te, y_te):
    y_pred = pipe.predict(X_te)
    return {
        'Accuracy_test': accuracy_score(y_te, y_pred),
        'Recall_test': sensitivity(y_te, y_pred),
        'Specificity_test': specificity(y_te, y_pred),
        'F1_test': f1_score_custom(y_te, y_pred)
    }


def make_original_pipeline(model_name, estimator_cls):
    if model_name == 'RandomForestClassifier':
        return Pipeline([('pre', preprocessor), ('clf', estimator_cls(random_state=42, class_weight='balanced'))])
    if model_name == 'ExtraTreesClassifier':
        return Pipeline([('pre', preprocessor), ('clf', estimator_cls(random_state=42, class_weight='balanced'))])
    if model_name == 'LogisticRegression':
        return Pipeline([('pre', preprocessor), ('clf', estimator_cls(max_iter=1000, random_state=42))])
    return Pipeline([('pre', preprocessor), ('clf', estimator_cls())])

comparison_rows = []
subgroup_reports = []

for model_name, (estimator_cls, _) in model_spaces.items():
    original_pipe = make_original_pipeline(model_name, estimator_cls)
    original_pipe.fit(X_train, y_train)
    original_metrics = evaluate_test_pipeline(original_pipe, X_test, y_test)
    comparison_rows.append({
        'Model': model_name,
        'Type': 'original',
        'Experiment': 'default',
        'Fitness_val': None,
        **original_metrics
    })

    best_key, best_value = max(results[model_name].items(), key=lambda item: item[1][1])
    best_params = best_value[0]
    best_score = best_value[1]
    best_pipe = build_pipeline(estimator_cls, best_params)
    best_pipe.fit(X_train, y_train)
    optimized_metrics = evaluate_test_pipeline(best_pipe, X_test, y_test)
    comparison_rows.append({
        'Model': model_name,
        'Type': 'optimized',
        'Experiment': best_key,
        'Fitness_val': best_score,
        **optimized_metrics
    })

    subgroup = subgroup_metrics(best_pipe, X_test, y_test, demographic_column)
    subgroup_reports.append((model_name, best_key, subgroup))

comparison_df = pd.DataFrame(comparison_rows)
print('Comparação de cada modelo original e otimizado:')
display(comparison_df)

for model_name, best_key, subgroup in subgroup_reports:
    print(f'\nSubgroup metrics for optimized {model_name} ({best_key}):')
    display(subgroup)

Comparação de cada modelo original e otimizado:


,Model,Type,Experiment,Fitness_val,Accuracy_test,Recall_test,Specificity_test,F1_test
0,RandomForestClassifier,original,default,NaN,0.536667,0.549342,0.523649,0.545752
1,RandomForestClassifier,optimized,exp3,0.536904,0.548333,0.608553,0.486486,0.577223
2,LogisticRegression,original,default,NaN,0.520000,0.582237,0.456081,0.551402
3,LogisticRegression,optimized,exp2,0.529701,0.518333,0.569079,0.466216,0.544882
4,KNeighborsClassifier,original,default,NaN,0.505000,0.516447,0.493243,0.513912
5,KNeighborsClassifier,optimized,exp3,0.553377,0.503333,0.526316,0.479730,0.517799
6,ExtraTreesClassifier,original,default,NaN,0.531667,0.539474,0.523649,0.538588
7,ExtraTreesClassifier,optimized,exp3,0.538368,0.523333,0.536184,0.510135,0.532680



Subgroup metrics for optimized RandomForestClassifier (exp3):


,group,recall,specificity,f1
0,F,0.613333,0.486111,0.582278
1,M,0.603896,0.486842,0.572308



Subgroup metrics for optimized LogisticRegression (exp2):


,group,recall,specificity,f1
0,F,0.593333,0.451389,0.559748
1,M,0.545455,0.480263,0.529968



Subgroup metrics for optimized KNeighborsClassifier (exp3):


,group,recall,specificity,f1
0,F,0.480000,0.458333,0.480000
1,M,0.571429,0.500000,0.553459



Subgroup metrics for optimized ExtraTreesClassifier (exp3):


,group,recall,specificity,f1
0,F,0.533333,0.486111,0.526316
1,M,0.538961,0.532895,0.538961


## Análise por subgrupos demográficos

Avalia se o modelo otimizado mantém desempenho consistente entre homens e mulheres.

In [18]:
print('A análise de comparação e subgroup metrics para todos os modelos otimizados já foi gerada na célula anterior.')

A análise de comparação e subgroup metrics para todos os modelos otimizados já foi gerada na célula anterior.


## Conclusões e próximos passos

- Revise o `param_space` para incluir outros hiperparâmetros relevantes de cada modelo.
- Aumente o número de gerações e população quando for rodar em ambiente com mais tempo.
- O notebook já avalia os modelos `RandomForestClassifier`, `LogisticRegression`, `KNeighborsClassifier` e `ExtraTreesClassifier`; adicione outros modelos da Fase 1 para expandir a análise.